In [3]:
# Imports
import pandas as pd
import numpy as np

# LOAD DELIVERABLES (ALREADY LAGGED)
df = pd.read_parquet("data/fx_system_df_lagged.parquet")
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# CONFIG
LOOKBACK = 21

# FX MAP
FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

# --- BUILD RAW 21D MOMENTUM PANEL ---
trend = pd.DataFrame(index=df.index)

for ccy in FX_TICKERS.keys():
    price = df[f"{ccy}_price"]
    trend[ccy] = np.log(price).diff(LOOKBACK)

# --- ALIGN ---
common_idx = trend.index.intersection(returns_df.index)
trend = trend.loc[common_idx]
returns_df = returns_df.loc[common_idx]

# --- VOL ADJUST ---
vol = returns_df.rolling(LOOKBACK).std().replace(0, np.nan)
trend = trend / vol

# --- COLLAPSE TO USD FACTOR ---
usd_trend = trend.mean(axis=1)

# --- CAUSAL NORMALIZATION ---
mean = usd_trend.rolling(LOOKBACK).mean()
std = usd_trend.rolling(LOOKBACK).std().replace(0, np.nan)
usd_trend = (usd_trend - mean) / std

# --- DISPERSION ---
disp = returns_df.std(axis=1).rolling(21).mean()

z = (disp - disp.rolling(LOOKBACK).mean()) / disp.rolling(LOOKBACK).std().replace(0, np.nan)

# --- INVERSE DISPERSION SCALAR ---
inv_gate = (-z).shift(1)
inv_gate = (1 + inv_gate).clip(lower=0, upper=2)

# --- APPLY SCALAR ---
usd_trend_scaled = usd_trend * inv_gate

# --- USD RETURNS ---
usd_returns = -returns_df.mean(axis=1)

# --- ALIGN ---
common_idx = usd_trend_scaled.index.intersection(usd_returns.index)
usd_trend_scaled = usd_trend_scaled.loc[common_idx]
usd_returns = usd_returns.loc[common_idx]

# --- STRATEGY ---
position = usd_trend_scaled.shift(1)
pnl = position * usd_returns
pnl_valid = pnl.dropna()

# --- SUMMARY ---
if pnl_valid.std() > 0:
    sharpe = pnl_valid.mean() / pnl_valid.std() * np.sqrt(252)
else:
    sharpe = np.nan

hit_rate = (pnl_valid > 0).mean()

results = {
    "LOOKBACK": 21,
    "Sharpe": sharpe,
    "Hit Rate": hit_rate,
    "Obs": len(pnl_valid)
}

results_df = pd.DataFrame([results])

# --- MAP TO CURRENCY SPACE ---
trend_usd_signal = pd.DataFrame(
    {ccy: -usd_trend_scaled for ccy in FX_TICKERS.keys()},
    index=usd_trend_scaled.index
)

# --- SAVE ---
trend_usd_signal.to_parquet("data/trend_usd_factor_signal.parquet")
pnl.to_frame("pnl").to_parquet("data/trend_usd_factor_pnl.parquet")
results_df.to_parquet("data/trend_usd_factor_summary.parquet")

results_df

,LOOKBACK,Sharpe,Hit Rate,Obs
0,21,0.111844,0.372848,5112
